# Notebook 08 — NLP Combos : extraction depuis transcription YouTube

**Objectif :** à partir d'une vidéo de gameplay/combo Yu-Gi-Oh, extraire automatiquement
les cartes mentionnées et modéliser la séquence comme un **graphe orienté de combo**.

**Pipeline :**
1. `youtube_transcript_api` → transcription gratuite sans clé API
2. Matching sur les noms de cartes de la DB (longest-match first)
3. Segmentation temporelle → séquences de combo
4. Graphe orienté A→B→C + visualisation pyvis

**Vidéo test :** https://www.youtube.com/watch?v=MsZb-dJAGHo

## Cell 1 — Imports & config

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
import sqlite3, re, os, json
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
import networkx as nx
from pyvis.network import Network
import warnings
warnings.filterwarnings('ignore')

DB = os.path.join('data', 'yugioh.db')
VIDEO_URL = 'https://www.youtube.com/watch?v=MsZb-dJAGHo'
VIDEO_ID  = VIDEO_URL.split('v=')[-1].split('&')[0]

# Fenêtre de combo : cartes mentionnées dans les X secondes suivantes = même séquence
COMBO_WINDOW_SEC = 30

print(f'Video ID : {VIDEO_ID}')
print(f'DB       : {DB}')

## Cell 2 — Charger les noms de cartes depuis la DB

In [ ]:
conn = sqlite3.connect(DB)

# Trier par longueur décroissante → longest-match first (évite de matcher 'Ash' avant 'Ash Blossom')
cards_df = pd.read_sql_query(
    'SELECT name FROM cards ORDER BY LENGTH(name) DESC',
    conn
)
conn.close()

card_names = cards_df['name'].tolist()

# Noms courts (<= 3 chars) = trop de faux positifs → on les filtre
card_names_filtered = [c for c in card_names if len(c) > 3]

print(f'{len(card_names_filtered)} cartes chargées (noms > 3 chars)')
print(f'Exemple long  : {card_names_filtered[0]}')
print(f'Exemple court : {card_names_filtered[-1]}')

# Pré-compiler les patterns regex (insensible à la casse)
# On échappe les caractères spéciaux dans les noms (ex: '&', '-', '/', '★')
card_patterns = [
    (name, re.compile(r'\b' + re.escape(name) + r'\b', re.IGNORECASE))
    for name in card_names_filtered
]
print(f'{len(card_patterns)} patterns compilés')

## Cell 3 — Récupérer la transcription YouTube

In [ ]:
def fetch_transcript(video_id: str) -> list[dict]:
    """
    Retourne la transcription sous forme de liste de segments :
    [{'text': '...', 'start': 12.4, 'duration': 2.1}, ...]
    Essaie d'abord EN, puis FR, puis la première dispo.
    """
    try:
        transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=['en'])
        print(f'Transcription EN récupérée ({len(transcript)} segments)')
        return transcript
    except NoTranscriptFound:
        pass
    try:
        transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=['fr'])
        print(f'Transcription FR récupérée ({len(transcript)} segments)')
        return transcript
    except NoTranscriptFound:
        pass
    try:
        # Prendre la première disponible
        transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
        t = next(iter(transcript_list))
        data = t.fetch()
        print(f'Transcription [{t.language_code}] récupérée ({len(data)} segments)')
        return data
    except Exception as e:
        print(f'Erreur : {e}')
        return []


transcript = fetch_transcript(VIDEO_ID)

if transcript:
    print(f'\nDurée totale estimée : {transcript[-1]["start"] + transcript[-1]["duration"]:.0f}s')
    print('\nExtrait (premiers segments) :')
    for seg in transcript[:8]:
        print(f'  [{seg["start"]:6.1f}s] {seg["text"]}')
else:
    print('Aucune transcription disponible pour cette vidéo.')

## Cell 4 — Extraction des cartes mentionnées

In [ ]:
def extract_card_mentions(transcript: list[dict], card_patterns: list) -> list[dict]:
    """
    Pour chaque segment de la transcription, cherche les noms de cartes.
    Retourne une liste de mentions : {card, start, text}
    Longest-match : une fois une carte trouvée dans le texte, on la retire pour éviter
    les sous-matches (ex: 'Snake-Eye' vs 'Snake-Eye Ash').
    """
    mentions = []
    for seg in transcript:
        text = seg['text']
        found_in_seg = set()
        remaining = text
        for card_name, pattern in card_patterns:   # déjà triés longest-first
            if pattern.search(remaining):
                found_in_seg.add(card_name)
                # Masquer pour éviter les sous-matches
                remaining = pattern.sub('', remaining)
        for card in found_in_seg:
            mentions.append({'card': card, 'start': seg['start'], 'text': seg['text']})
    return mentions


if transcript:
    mentions = extract_card_mentions(transcript, card_patterns)
    mentions_df = pd.DataFrame(mentions).sort_values('start').reset_index(drop=True)

    print(f'{len(mentions_df)} mentions de cartes détectées')
    print(f'{mentions_df["card"].nunique()} cartes uniques\n')

    # Top cartes les plus mentionnées
    top_cards = mentions_df['card'].value_counts().head(20)
    print('Top 20 cartes les plus mentionnées :')
    print(top_cards.to_string())
else:
    mentions_df = pd.DataFrame(columns=['card', 'start', 'text'])
    print('Pas de transcription — mentions vides.')

## Cell 5 — Segmentation en séquences de combo

In [ ]:
def segment_combos(mentions_df: pd.DataFrame, window_sec: float = 30) -> list[list[str]]:
    """
    Regroupe les mentions en séquences de combo :
    si deux mentions sont à moins de `window_sec` secondes, elles font partie du même combo.
    Retourne une liste de séquences (listes de noms de cartes ordonnées dans le temps).
    """
    if mentions_df.empty:
        return []
    combos = []
    current = []
    last_time = -999
    for _, row in mentions_df.iterrows():
        if row['start'] - last_time > window_sec and current:
            combos.append(current)
            current = []
        current.append(row['card'])
        last_time = row['start']
    if current:
        combos.append(current)
    return combos


combos = segment_combos(mentions_df, window_sec=COMBO_WINDOW_SEC)

print(f'{len(combos)} séquences de combo détectées (fenêtre = {COMBO_WINDOW_SEC}s)\n')

# Afficher les plus longues séquences
combos_sorted = sorted(combos, key=len, reverse=True)
print('Top 5 séquences les plus longues :')
for i, combo in enumerate(combos_sorted[:5]):
    print(f'  Combo {i+1} ({len(combo)} cartes) : {" → ".join(combo[:8])}{"..." if len(combo)>8 else ""}')

## Cell 6 — Graphe orienté de combos

In [ ]:
def build_combo_graph(combos: list[list[str]]) -> nx.DiGraph:
    """
    Construit un graphe orienté où une arête A→B signifie
    que la carte B est jouée après la carte A dans au moins une séquence de combo.
    Le poids de l'arête = nombre de fois que la transition A→B apparaît.
    """
    G = nx.DiGraph()
    for combo in combos:
        for i in range(len(combo) - 1):
            a, b = combo[i], combo[i+1]
            if a == b:
                continue
            if G.has_edge(a, b):
                G[a][b]['weight'] += 1
            else:
                G.add_edge(a, b, weight=1)
    return G


G = build_combo_graph(combos)

print(f'Graphe de combo : {G.number_of_nodes()} nœuds, {G.number_of_edges()} arêtes')

# Top transitions les plus fréquentes
if G.number_of_edges() > 0:
    edges = sorted(G.edges(data=True), key=lambda e: e[2]['weight'], reverse=True)
    print('\nTop 15 transitions A→B :')
    for a, b, d in edges[:15]:
        print(f'  {a} → {b}  (×{d["weight"]})')

# Nœuds centraux (in-degree = cartes vers lesquelles on joue le plus)
if G.number_of_nodes() > 0:
    in_deg = sorted(G.in_degree(), key=lambda x: x[1], reverse=True)[:5]
    out_deg = sorted(G.out_degree(), key=lambda x: x[1], reverse=True)[:5]
    print('\nCartes "destination" (in-degree élevé) :', [n for n,_ in in_deg])
    print('Cartes "source"      (out-degree élevé) :', [n for n,_ in out_deg])

## Cell 7 — Visualisation pyvis du graphe

In [ ]:
def visualize_combo_graph(G: nx.DiGraph, output_path: str, title: str = 'Combo Graph') -> None:
    if G.number_of_nodes() == 0:
        print('Graphe vide — pas de visualisation.')
        return

    # Filtrer les arêtes faibles (garder weight >= 2 si beaucoup d'arêtes)
    min_weight = 2 if G.number_of_edges() > 30 else 1
    G_filtered = nx.DiGraph()
    for a, b, d in G.edges(data=True):
        if d['weight'] >= min_weight:
            G_filtered.add_edge(a, b, weight=d['weight'], title=f'×{d["weight"]}')

    if G_filtered.number_of_nodes() == 0:
        G_filtered = G  # fallback : garder tout

    net = Network(height='700px', width='100%', directed=True, bgcolor='#1a1a2e', font_color='white')
    net.set_options("""{
      \"physics\": {\"enabled\": true, \"stabilization\": {\"iterations\": 200}},
      \"edges\": {\"arrows\": {\"to\": {\"enabled\": true}}, \"smooth\": {\"type\": \"curvedCW\", \"roundness\": 0.2}},
      \"nodes\": {\"font\": {\"size\": 12}}
    }""")

    # Taille des nœuds ∝ degré total
    degree = dict(G_filtered.degree())
    max_deg = max(degree.values()) if degree else 1
    in_deg_map = dict(G_filtered.in_degree())

    for node in G_filtered.nodes():
        size = 15 + 25 * (degree.get(node, 0) / max_deg)
        # Couleur selon in-degree : rouge = "destination finale", bleu = "starter"
        id_ratio = in_deg_map.get(node, 0) / (degree.get(node, 1))
        color = f'hsl({int(240 - id_ratio * 200)}, 80%, 55%)'
        net.add_node(node, label=node, size=size, color=color, title=f'Degree: {degree[node]}')

    for a, b, d in G_filtered.edges(data=True):
        width = 1 + d.get('weight', 1) * 1.5
        net.add_edge(a, b, width=width, title=d.get('title', ''))

    net.save_graph(output_path)
    print(f'Graphe sauvegardé : {output_path}')


video_id_safe = VIDEO_ID.replace('-', '_')
output_path = f'data/graph_combo_{video_id_safe}.html'
visualize_combo_graph(G, output_path, title=f'Combo Graph — {VIDEO_ID}')

# Afficher dans le notebook
from IPython.display import IFrame, display
if os.path.exists(output_path):
    display(IFrame(src=output_path, width='100%', height='720px'))

## Cell 8 — Sauvegarde en DB + résumé

In [ ]:
# Sauvegarder les mentions et transitions en DB
conn = sqlite3.connect(DB)

# Table combo_mentions : toutes les cartes détectées par vidéo
if not mentions_df.empty:
    mentions_out = mentions_df.copy()
    mentions_out['video_id'] = VIDEO_ID
    mentions_out.to_sql('combo_mentions', conn, if_exists='append', index=False)
    print(f'combo_mentions : {len(mentions_out)} lignes insérées')

# Table combo_edges : transitions A→B avec poids
if G.number_of_edges() > 0:
    edges_rows = [
        {'video_id': VIDEO_ID, 'card_from': a, 'card_to': b, 'weight': d['weight']}
        for a, b, d in G.edges(data=True)
    ]
    edges_df = pd.DataFrame(edges_rows)
    edges_df.to_sql('combo_edges', conn, if_exists='append', index=False)
    print(f'combo_edges : {len(edges_df)} arêtes insérées')

conn.commit()
conn.close()

print('\n=== RÉSUMÉ ===')
print(f'Vidéo        : {VIDEO_URL}')
print(f'Segments     : {len(transcript)}')
print(f'Mentions     : {len(mentions_df)} ({mentions_df["card"].nunique() if not mentions_df.empty else 0} cartes uniques)')
print(f'Combos       : {len(combos)} séquences')
print(f'Graphe       : {G.number_of_nodes()} nœuds, {G.number_of_edges()} arêtes')
print(f'Visualisation: {output_path}')

## Cell 9 — Étendre à plusieurs vidéos

Pour analyser plusieurs vidéos automatiquement (sans YouTube API key), 
il suffit de lister les video IDs et reboucler.

```python
VIDEO_IDS = [
    'MsZb-dJAGHo',  # vidéo test
    # Ajouter d'autres IDs ici
]

all_mentions = []
all_combos   = []

for vid in VIDEO_IDS:
    t = fetch_transcript(vid)
    if not t: continue
    m = extract_card_mentions(t, card_patterns)
    m_df = pd.DataFrame(m).sort_values('start')
    c = segment_combos(m_df, COMBO_WINDOW_SEC)
    all_mentions.extend(m)
    all_combos.extend(c)
    print(f'{vid}: {len(m)} mentions, {len(c)} combos')

# Graphe agrégé multi-vidéos
G_global = build_combo_graph(all_combos)
visualize_combo_graph(G_global, 'data/graph_combo_global.html')
```

**Prochaines sources possibles :**
- Recherche manuelle de IDs sur YouTube (`DoomZ combo guide 2026`)
- Canaux connus : TeamSamuraiX1, Farfa, Rata, NimbleSummoner
- Avec une clé YouTube Data API → `search` automatique par archetype